# 12 · Facility siting and equity tradeoffs

**Question:** Does the best average travel time also provide the fairest access?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
All city geometry, residents, sensor observations and parameters are **synthetic educational fixtures**.
The longitude/latitude anchor is near Gaborone, but these are not mapped Gaborone assets or a validated city twin.

**Astra experiment:** Give Astra the candidate table and an explicit objective, require a decision with numerical evidence, then verify it by exhaustive enumeration.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## Choose an objective · 30 minutes
**Learn:** separate efficiency from coverage, examine geographic groups, and audit an optimization recommendation.
The invented city has 1,200 residents. West/east groups describe location only, not ethnicity,
income or disease status. A barrier has one northern crossing; all edges are bidirectional.
Choose one additional facility. We compare minimum population-weighted mean time with maximum
worst-group coverage within `threshold_min`. Neither objective represents service capacity or quality.


In [ ]:
from experiments import access_summary, grade_answer
c=city(); pop=allocate(c['capacity'],1200)
nodes,graph=network(blocked=[(r*11+5,r*11+6) for r in range(10)])
idx,connector=snap(c['xy'],nodes)
groups={'west':c['xy'][:,0]<1100,'east':c['xy'][:,0]>=1100}
clinic=22; speed_m_min=80.; threshold_min=15.
candidate_nodes=[6,28,50,72,94,116]
rows=[]; candidate_times={}
for candidate in candidate_nodes:
    minutes=(shortest(graph,[clinic,candidate])[idx]+connector)/speed_m_min
    candidate_times[candidate]=minutes
    overall=access_summary(minutes,pop,threshold_min)
    coverage={name:access_summary(minutes[mask],pop[mask],threshold_min)['coverage'] for name,mask in groups.items()}
    rows.append(dict(node=candidate,mean_min=overall['mean_reachable_min'],coverage=overall['coverage'],
                     worst_group_coverage=min(coverage.values()),group_coverage=coverage,
                     unreachable_population=overall['unreachable_population']))
efficiency=min(rows,key=lambda r:(r['mean_min'],r['node']))
equity=min(rows,key=lambda r:(-r['worst_group_coverage'],r['mean_min'],r['node']))
assert all(r['unreachable_population']==0 for r in rows)
assert all(0<=r['worst_group_coverage']<=r['coverage']<=1 for r in rows)
print('node | mean minutes | west coverage | east coverage')
for r in rows: print(r['node'],round(r['mean_min'],3),{k:round(v,3) for k,v in r['group_coverage'].items()})
print('Efficiency choice:',efficiency['node'],'| equity choice:',equity['node'])


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
for u,edges in graph.items():
    for v,w in edges:
        if u<v: axes[0].plot(*nodes[[u,v]].T,color='#bbc5cc',lw=.6)
for name,mask in groups.items(): axes[0].scatter(*c['xy'][mask].T,s=15,label=name)
axes[0].scatter(*nodes[clinic],marker='P',s=130,c='black',label='Existing')
for node in candidate_nodes:
    axes[0].scatter(*nodes[node],marker='*',s=130,c='#b34f00'); axes[0].annotate(str(node),nodes[node])
map_axes(axes[0],'Synthetic geographic groups'); axes[0].legend()
axes[1].scatter([r['mean_min'] for r in rows],[r['worst_group_coverage'] for r in rows],s=90)
for r in rows: axes[1].annotate(str(r['node']),(r['mean_min'],r['worst_group_coverage']),xytext=(5,5),textcoords='offset points')
axes[1].set(xlabel='Population-weighted mean minutes (lower is better)',ylabel='Worst-group coverage (higher is better)',title='Every candidate, two objectives',ylim=(0,1.05))
plt.tight_layout(); plt.show()


## Ask Astra to make a bounded decision
Send only `12_prompt.json`. Its candidate table is evidence, not a preselected answer.
The objective and tie-breaks are declared before evaluation. A model should not substitute a different objective.
Discuss whether group boundaries or travel thresholds change the recommendation; that is a policy choice,
not something the model can resolve from coordinates alone.


In [ ]:
prompt=dict(task='Choose exactly one candidate maximizing worst_group_coverage; break ties by lowest mean_min, then lowest node ID. Return ONLY JSON with chosen_node and worst_group_coverage.',
            synthetic=True,threshold_min=threshold_min,coverage_units='fraction of all group residents',
            candidates=rows,assumptions=['No capacity constraint','Local synthetic groups','All residents reachable'])
export_json('12_prompt.json',prompt)
expected=dict(chosen_node=equity['node'],worst_group_coverage=equity['worst_group_coverage'])
response_source='FIXTURE - efficiency objective substituted for equity'
model_name='not run'; trial_date='not run'; interface='fixture'
response_text=json.dumps(dict(chosen_node=efficiency['node'],worst_group_coverage=efficiency['worst_group_coverage']))
score=grade_answer(response_text,expected,{'worst_group_coverage':.001})
display(score)
export_json('12_trial.json',dict(response_source=response_source,model=model_name,date=trial_date,
            interface=interface,prompt=prompt,response_text=response_text,score=score,reference=expected,
            coverage_tolerance=.001))


## Check a misleading denominator
A disconnected person must count as unserved. Dropping disconnected residents from both the numerator
and denominator can make a damaged network appear to perform better.
**Try:** set the service threshold to 10 or 20 minutes. Do the efficiency and equity choices still differ?
Agreement for some parameters is a useful result too; do not force a tradeoff.


In [ ]:
tiny=access_summary([5.,float('inf')],[60,40],15.)
assert tiny['coverage']==.6 and tiny['unreachable_population']==40
assert tiny['mean_reachable_min']==5.
print('60 served + 40 disconnected: coverage =',tiny['coverage'],'not 100%.')


## Extend with Codex or Astra
Add a capacity-limited assignment with an explicitly unserved category. Preserve population conservation, test that no facility exceeds capacity, and report objective changes separately from solver errors.

Keep the baseline seed and tests fixed while changing one assumption. Record the model name,
date, exact prompt, response and error metrics. Generated code must pass the same checks.
Download files in `exports/` and your edited notebook; browser storage does not commit changes to GitHub.
See [the project guide](https://github.com/jltobias/JupyterLite-GPT-6-Astra-Geospatial/blob/main/docs/GUIDE.md)
for capabilities, evidence, sources and the route to real data.
